# Task 3: The Social Planner Test

Solve the planner's welfare-maximisation problem in cvxpy, pull the dual variables of the resource constraints, and compare them to the analytical market prices.

See the Week 2 report for the full derivation and interpretation; this notebook just runs the experiment.

In [6]:
import numpy as np
import cvxpy as cp

np.set_printoptions(precision=4, suppress=True)

## 1. Economy parameters

Three consumers, two goods. `alpha[j, k]` is consumer $j$'s Cobb-Douglas weight (spending share) on good $k$; rows sum to 1.

In [7]:
alpha = np.array([
    [0.6, 0.4],   # consumer A
    [0.3, 0.7],   # consumer B
    [0.5, 0.5],   # consumer C
])
w = np.array([100.0, 80.0, 60.0])   # wealth of each consumer
E = np.array([30.0, 40.0])          # endowment of each good

N, K = alpha.shape

## 2. Analytical market-clearing prices

Aggregating the Cobb-Douglas demand from Task 1 gives the closed form
$$p_k^{\text{market}} = \frac{\sum_j \alpha_{jk}\, w_j}{E_k}.$$
These are the ground truth we will compare the planner's duals against.

In [8]:
p_market = (alpha * w[:, None]).sum(axis=0) / E
x_market = alpha * w[:, None] / p_market[None, :]

def norm(p):
    return p / p.sum()

print("market prices (raw):       ", p_market)
print("market prices (normalised):", norm(p_market))

market prices (raw):        [3.8  3.15]
market prices (normalised): [0.5468 0.4532]
market prices (raw):        [3.8  3.15]
market prices (normalised): [0.5468 0.4532]


## 3. The planner solver

Maximise a weighted sum of utilities subject to the two resource constraints. The duals of those constraints are the shadow prices we will compare to the market.

In [9]:
def solve_planner(theta):
    x = cp.Variable((N, K), nonneg=True)
    welfare = sum(
        theta[j] * (alpha[j, 0] * cp.log(x[j, 0])
                    + alpha[j, 1] * cp.log(x[j, 1]))
        for j in range(N)
    )
    resource = [cp.sum(x[:, k]) <= E[k] for k in range(K)]
    problem = cp.Problem(cp.Maximize(welfare), resource)
    problem.solve()
    duals = np.array([c.dual_value for c in resource])
    return x.value, duals, problem.value

## 4. Equal weights vs Negishi weights

Run the planner twice: once with $\theta_j = 1$ (equal weights, the egalitarian planner) and once with $\theta_j = w_j$ (Negishi weights).

In [10]:
_, p_equal,   _ = solve_planner(np.ones(N))
_, p_negishi, _ = solve_planner(w)

print(f"equal-weight  duals (normalised): {norm(p_equal)}")
print(f"Negishi       duals (normalised): {norm(p_negishi)}")
print(f"market prices       (normalised): {norm(p_market)}")
print()
print(f"gap, equal-weight vs market: {np.abs(norm(p_equal)   - norm(p_market))}")
print(f"gap, Negishi     vs market: {np.abs(norm(p_negishi) - norm(p_market))}")

equal-weight  duals (normalised): [0.5385 0.4615]
Negishi       duals (normalised): [0.5468 0.4532]
market prices       (normalised): [0.5468 0.4532]

gap, equal-weight vs market: [0.0083 0.0083]
gap, Negishi     vs market: [0. 0.]
equal-weight  duals (normalised): [0.5385 0.4615]
Negishi       duals (normalised): [0.5468 0.4532]
market prices       (normalised): [0.5468 0.4532]

gap, equal-weight vs market: [0.0083 0.0083]
gap, Negishi     vs market: [0. 0.]


## Conclusion

- Equal weights: duals disagree with market prices by about $0.8\%$ per component. Wrong ratio, not just wrong scale.
- Negishi weights ($\theta = w$): duals reproduce the market prices exactly, to solver tolerance.
- **Market-clearing prices = shadow prices of the wealth-weighted planner.** The market and the planner are two descriptions of the same equilibrium.